In [0]:
# Databricks notebook source
# Creates the audit log table, the quarantine table and the logging helper.
# Safe to run many times (CREATE TABLE IF NOT EXISTS).
 
import uuid
from datetime import datetime
 
# COMMAND ----------
 
# 1. One row per pipeline run (per layer)
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb.pipeline_execution_logs (
    log_id               STRING,
    layer_processed      STRING,     -- 'Raw-to-Bronze' or 'Bronze-to-Silver'
    load_type            STRING,     -- 'full' or 'incremental'
    batch_id             STRING,
    parameter_processed  STRING,     -- file/folder path or batch ids that were processed
    execution_start_time TIMESTAMP,
    execution_end_time   TIMESTAMP,
    status               STRING,     -- 'SUCCESS' or 'FAILURE'
    rows_inserted        LONG,
    rows_updated         LONG,
    rows_quarantined     LONG,
    error_message        STRING
)
USING DELTA
""")
 
# 2. Bad / non-conforming records from any layer
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb.quarantine_movies (
    layer           STRING,
    batch_id        STRING,
    source_file     STRING,
    reject_reason   STRING,
    raw_record      STRING,
    load_timestamp  TIMESTAMP
)
USING DELTA
""")
 
# COMMAND ----------
 
def log_pipeline_execution(layer, load_type, batch_id, parameter, start_time, end_time,
                           status, rows_inserted=0, rows_updated=0, rows_quarantined=0,
                           error_msg="NONE"):
    """Write one row to pipeline_execution_logs."""
    row = [(
        str(uuid.uuid4()), layer, load_type, batch_id, str(parameter),
        start_time, end_time, status,
        int(rows_inserted), int(rows_updated), int(rows_quarantined),
        str(error_msg)[:2000],
    )]
    schema = spark.table("workspace.tmdb.pipeline_execution_logs").schema
    spark.createDataFrame(row, schema).write.format("delta").mode("append") \
        .saveAsTable("workspace.tmdb.pipeline_execution_logs")
    print(f"LOGGED {layer} | {load_type} | batch={batch_id} | {status} | "
          f"inserted={rows_inserted} updated={rows_updated} quarantined={rows_quarantined}")
 